# 练习实验：分块 (Chunking)
---
欢迎来到关于分块（Chunking）的练习实验！正如你在课程中所见，分块将长文本分解为更小、易于处理的片段，这对于高效地使用向量数据库和语言模型至关重要。

# 目录
- [ 1 - 简介](#1)
  - [ 1.1 导入必要的库](#1-1)
  - [ 1.2 下载数据](#1-2)
- [ 2 - 固定大小分块](#2)
  - [ 2.1 分块代码示例](#2-1)
  - [ 2.2 带重叠的分块](#2-2)
- [ 3 - 可变大小分块 - 递归字符拆分](#3)
  - [ 3.1 可变大小分块方法的伪代码](#3-1)
  - [ 3.2 混合固定和可变大小的分块](#3-2)
- [ 4 - 对真实数据进行分块](#4)
  - [ 4.1 获取数据](#4-1)
  - [ 4.2 章节分块](#4-2)
  - [ 4.3 将分块加载到向量数据库中](#4-3)
- [ 5 - 搜索 ](#5)
- [ 6 - 集成到 RAG 系统中](#6)

<a id='1'></a>
## 1 - 简介

---

分块在信息检索中发挥着重要作用。例如，在从一系列书籍构建向量数据库时，不同的分块大小可以服务于不同的目的。将整本书编目为单个向量可能有助于识别宏观主题，但会遗漏具体细节。而在段落或句子级别进行分块，则可以检索到特定的信息或概念。

语言模型通常对一次能处理的文本量有限制，这被称为“上下文窗口”。分块有助于确保文本输入保持在这些界限内，从而允许模型通过将小说等大型文档拆分为较小的部分来处理它们。

在这个练习实验中，你将探索不同的分块方式，并了解它们如何影响 RAG 系统！


<div align="center">
  <img src="images/chunking.png" alt="概览" width="80%">
</div>

<a id='1-1'></a>
### 1.1 导入必要的库

In [1]:
from typing import List
import requests
import re
import weaviate
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.util import generate_uuid5
import tqdm
from weaviate.classes.query import Filter


In [2]:
# 从自定义的工具包 utils 中导入核心功能函数
from utils import (
    generate_with_single_input, # 用于调用大模型进行文本生成的函数
    suppress_subprocess_output,  # 用于隐藏子进程产生的冗余日志输出
    kill_processes_on_ports     # 用于强制清理（杀死）指定端口上的占用进程
)

# 在导入并启动 Flask 应用之前，先清理可能存在的残留进程
# 警告：如果该单元格在同一个会话中运行两次，由于它会扫描并关闭端口，
# 某些环境下可能会误伤正在运行的代码内核（Kernel），导致连接断开。
# kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])
# 作用：确保以下关键端口是“干净”的：
# 5000：通常用于 Flask 推理服务（你的 BGE 向量模型）
# 8080 / 50051：通常用于 Weaviate 数据库的 REST 和 gRPC 通信

# 导入 flask_app 模块
import flask_app
# 作用：这行代码通常会触发 flask_app/__init__.py 或内部的启动逻辑，
# 正式在后台开启向量化推理服务和重排序服务。

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


 * Serving Flask app 'flask_app'
 * Debug mode: off


<a id='1-2'></a>
### 1.2 下载数据

现在，你需要一段足够长的文本，以便进行分块处理。让我们从 [Pro Git 书籍](https://git-scm.com/book/en/v2) 中截取一部分，具体是名为“什么是 Git？”的章节。

In [3]:
# 定义目标文件的远程 URL 地址
# 这里指向的是《Pro Git》开源书中的一个章节文件（Asciidoc 格式）
url = "https://raw.githubusercontent.com/progit/progit2/main/book/01-introduction/sections/what-is-git.asc"

# 使用 requests 库发送 HTTP GET 请求，并获取返回的文本内容
source_text = requests.get(url).text
# 作用：程序会访问该网址，下载其背后的原始字符流，
# 并将其赋值给变量 source_text，供后续的文本处理或模型训练使用。

In [4]:
# 打印 source_text 变量中的前 1000 个字符
print(source_text[:1000])
# 作用：
# 1. 验证抓取：确认 requests.get() 是否成功拿到了数据，而不是返回了 404 错误页面。
# 2. 观察格式：了解文档的结构（例如：它是 Markdown 还是 Asciidoc？开头是否有元数据？）。
# 3. 检查编码：观察中文字符或特殊符号是否显示正常，确保没有乱码。

[[what_is_git_section]]
=== What is Git?

So, what is Git in a nutshell?
This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.
As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.
Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))

==== Snapshots, Not Differences

The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.
Conceptually, most other systems store information as a list of file-based changes.
These other systems (CVS, Subversion, Perforce, and so o

In [4]:
# 使用格式化字符串（f-string）打印字数与 Token 预估值
print(f"There are about {len(source_text.split())} words in this chapter. Depending on how your LLM tokenizes words, you'd expect roughly {round(len(source_text.split())*1.3)} tokens.")

# 逻辑拆解：
# 1. len(source_text.split()): 
#    作用：通过空格将文本拆分成单词列表并计算长度。这是对英文文本最简单的“字数统计”。

# 2. *1.3: 
#    原因：LLM（如 GPT-4, Llama）并不直接读取单词，而是读取 Token（词元）。
#    在英文中，1 个单词通常对应约 1.3 到 1.4 个 Token。

# 3. round(...): 
#    作用：对计算结果进行四舍五入取整，得到一个直观的预估数值。

There are about 1403 words in this chapter. Depending on how your LLM tokenizes words, you'd expect roughly 1824 tokens.


<a id='2'></a>
## 2 - 固定大小分块
---
固定大小分块意味着将文本分解为相同大小的片段。例如，你可以将一篇文章拆分为每份 100 个单词的部分，或者每份 200 个字符的章节。这种方法很常用，因为它简单易用且效果良好。

它的工作原理是将文本划分为具有固定单位数量的片段。这些单位可以是单词、字符，甚至是 token。每个片段中的单位数量在最高上限内保持一致，并且片段之间可以包含可选的重叠部分。


<div align="center">
  <img src="images/fixed_size.png" alt="固定大小分块" width="80%">
</div>

<a id='2-1'></a>
### 2.1 分块代码示例

现在让我们看一个固定大小分块的实现方案。实现方式有很多种，以下是其中一种可能的实现。

In [5]:
def get_chunks_fixed_size(text: str, chunk_size: int) -> List[str]:
    """
    将给定文本按指定的固定字数拆分成块。

    参数:
        text (str): 要拆分的输入文本。
        chunk_size (int): 每个块包含的最大单词数。

    返回:
        List[str]: 文本块列表，每个块包含最多 'chunk_size' 个单词。
    """
    # 将输入文本按空格拆分为单个单词列表
    text_words = text.split()
    
    # 初始化一个列表，用于存放拆分后的单词块
    chunks = []
    
    # 以 chunk_size 为步长，遍历单词索引
    for i in range(0, len(text_words), chunk_size):
        # 截取从 i 到 i + chunk_size 的单词子列表
        chunk_words = text_words[i: i + chunk_size]
        
        # 将选中的单词重新用空格连接成一个完整的字符串
        chunk = " ".join(chunk_words)
        
        # 将生成的字符串块添加到列表中
        chunks.append(chunk)
    
    # 返回最终的文本块列表
    return chunks

In [6]:
# 调用函数，将 source_text 拆分为每块包含 100 个单词的列表
fixed_size_chunks = get_chunks_fixed_size(source_text, chunk_size = 100)

# 执行结果：
# 1. fixed_size_chunks 现在是一个字符串列表 (List[str])。
# 2. 列表中的每一个元素（块）长度约为 100 个单词。
# 3. 如果原始内容有 1000 个单词，那么这个列表大约会有 10 个元素。

In [7]:
# 打印分块后的列表长度
print(len(fixed_size_chunks))

# 作用与意义：
# 1. 确认产出：如果你看到数字是 0，说明之前的 source_text 抓取或切分逻辑出了问题。
# 2. 规模预估：这个数字直接决定了你接下来要往向量数据库（Weaviate）中写入多少个对象。
# 3. 颗粒度评估：例如，如果文章有 800 个词，设置 chunk_size=100，这里应该输出 8。

15


In [8]:
# 访问列表中的前三个分块（索引为 0, 1, 2）
fixed_size_chunks[0:3]

# 作用与意义：
# 1. 质量抽检：观察前三个块的内容，确认它们是否在合理的位置被切断。
# 2. 格式确认：确认原始文本中的特殊符号（如 Git 指令中的 $ 符号或 Asciidoc 标记）是否被完整保留。
# 3. 颗粒度感知：直观感受“100 个单词”到底有多长，判断这是否适合作为 LLM 的检索上下文。

["[[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool. Even though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in",
 'a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce))) ==== Snapshots, Not Differences The major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data. Conceptually, most other systems store information as a list of file-based changes. These other systems (CVS, Subversion, Perforce, and s

<a id='2-2'></a>
### 2.2 带重叠的分块

让我们修改代码以允许重叠，这样分块之间将拥有共享的 token。


<div align="center">
  <img src="images/overlap.png" alt="带重叠的分块" width="80%">
</div>

In [9]:
def get_chunks_fixed_size_with_overlap(text: str, chunk_size: int, overlap_fraction: float) -> List[str]:
    """
    将给定文本拆分为固定大小的块，并在连续块之间设置指定的重叠比例。

    参数:
    - text (str): 输入的长文本。
    - chunk_size (int): 每个块包含的单词数量。
    - overlap_fraction (float): 重叠部分占块大小的比例（如 0.2 表示重叠 20%）。

    返回:
    - List[str]: 包含重叠内容的文本块列表。
    """

    # 将文本按空格拆分为单词列表
    text_words = text.split()
    
    # 计算连续块之间需要重叠的单词数量（整数）
    # 例如：100词的块，0.2的比例，意味着重叠 20 个单词
    overlap_int = int(chunk_size * overlap_fraction)
    
    # 初始化列表，用于存储最终的文本块
    chunks = []
    
    # 以 chunk_size 为步长遍历文本
    for i in range(0, len(text_words), chunk_size):
        # 核心逻辑：确定当前块的起点和终点
        # 起点使用了 max(i - overlap_int, 0)，这意味着除了第一个块外，
        # 后续每个块都会“往回看”，抓取上一个块末尾的单词。
        start_index = max(i - overlap_int, 0)
        end_index = i + chunk_size
        
        # 根据计算好的索引截取单词
        chunk_words = text_words[start_index: end_index]
        
        # 将单词重新拼接成字符串
        chunk = " ".join(chunk_words)
        
        # 将该块存入列表
        chunks.append(chunk)
    
    # 返回最终的分块结果
    return chunks

In [10]:
# 遍历不同的分块大小：5词（极小）、25词（中等）、100词（较大）
for chosen_size in [5, 25, 100]:
    # 调用带重叠的分块函数，固定重叠比例为 20%
    # 例如：当 size 为 100 时，每个块会包含上一个块末尾的 20 个单词
    chunks = get_chunks_fixed_size_with_overlap(source_text, chosen_size, overlap_fraction=0.2)
    
    # 在屏幕上打印当前测试的元数据
    print(f"\n分块大小: {chosen_size} - 返回了 {len(chunks)} 个分块。")
    
    # 循环打印每个测试组的前 3 个分块，以便观察切分效果和重叠内容
    for i in range(3):
        # 打印分块编号及其具体的文本内容
        print(f"分块 {i+1}: {chunks[i]}")


分块大小: 5 - 返回了 281 个分块。
分块 1: [[what_is_git_section]] === What is Git?
分块 2: Git? So, what is Git in
分块 3: in a nutshell? This is an

分块大小: 25 - 返回了 57 个分块。
分块 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git
分块 2: if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to
分块 3: you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid

分块大小: 100 - 返回了 15 个分块。
分块 1: [[what_is_git_section]] === What is Git? So, what is Git in a nutshell? This is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you. As you learn Git, try to clear your mind of the things you may know about other VCSs,

请注意，较小的文本块虽然非常详细，但它们可能**没有足够的信息来有效地进行搜索**。相比之下，**较大的分块开始包含更多信息，长度类似于典型的段落**。随着这些分块变得更长，**它们相关的向量嵌入会变得更加笼统**。最终，它们会达到一个点，即不再有效地用于信息搜索。

<a id='3'></a>
## 3 - 可变大小分块 - 递归字符拆分

---
现在让我们探讨一下可变大小分块。与固定大小分块不同，这里每个分块的大小是处理后的结果，而不是预设的起点。在可变大小分块中，文本是使用特定的标记进行划分的。这种标记可以是句子或段落的分隔符，甚至可以是像 Markdown 标题这样的结构性元素。

<div align="center">
  <img src="images/recursive.png" alt="递归字符拆分" width="80%">
</div>

<a id='3-1'></a>
### 3.1 可变大小分块方法的伪代码

最简单的一种是将其拆分为段落（`\n\n`）

In [11]:
# 定义一个按段落拆分文本的函数
def get_chunks_by_paragraph(source_text: str) -> List[str]:
    # 使用双换行符 "\n\n" 作为分隔符进行拆分
    # 在大多数 Markdown、Asciidoc 或纯文本文件中，双换行符是段落之间的标准界限
    return source_text.split("\n\n")

# 作用：
# 1. 语义保真：确保每一个分块都是一个完整的表达，不会出现“话说到一半被切断”的情况。
# 2. 结构对齐：对于《Pro Git》这种高质量书籍，段落通常是围绕一个单一主题展开的。

另一种方法，在这种情况下，是将其拆分为章节。正如你在检查文本时所见，章节是使用 `\n==` 标记进行分隔的。

In [12]:
# 定义一个按 Asciidoc 标题标记拆分文本的函数
def get_chunks_by_asciidoc_sections(source_text: str) -> List[str]:
    # 在 Asciidoc 格式中，"==" 通常表示二级标题（Section）
    # 使用 "\n==" 作为分隔符，可以将文档拆分为一个个独立的章节或子节
    return source_text.split("\n==")

# 作用：
# 1. 逻辑高度统一：每一个分块都代表了一个完整的技术主题。
# 2. 检索上下文极佳：当用户搜索“Git 的基本原理”时，系统能直接返回整个章节。

In [13]:
# 遍历两种不同的结构化分隔符：段落 vs 章节
for marker in ["\n\n", "\n=="]:
    # 根据当前标记对 source_text 进行拆分
    chunks = source_text.split(marker)
    
    # 打印测试元数据，使用 repr() 是为了能清晰地看到 \n 等转义字符
    print(f"\n使用分隔符: {repr(marker)} - 返回了 {len(chunks)} 个分块。")
    
    # 打印前 3 个分块进行对比
    for i in range(3):
        # 再次使用 repr()，这样可以观察到分块开头和结尾是否包含了多余的空格或换行
        print(f"分块 {i+1}: {repr(chunks[i])}")


使用分隔符: '\n\n' - 返回了 31 个分块。
分块 1: '[[what_is_git_section]]\n=== What is Git?'
分块 2: "So, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))"
分块 3: '==== Snapshots, Not Differences'

使用分隔符: '\n==' - 返回了 7 个分块。
分块 1: '[[what_is_git_section]]'
分块 2: "= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundam

简单基于标记的分块的一个明显问题是，**标题经常会变成独立的分块**，这可能并不理想。在实践中，你可以采用混合策略，将简短的分块（如标题）附加到随后的分块中。这样，标题就能与其相关的部分保持联系。让我们进一步探索这种方法。

<a id='3-2'></a>
### 3.2 混合固定和可变大小的分块

你可以结合固定大小和可变大小的分块方法，以利用这两者的优势。例如，使用可变大小的分块器在段落标记处划分文本，然后应用固定大小的过滤器。如果一个分块太小，你可以将其与下一个分块合并；如果一个分块太大，你可以在中间或在该分块内的另一个标记处进行拆分。

In [14]:
def mixed_chunking(source_text):
    """
    结合固定大小和变量大小的逻辑对文本进行切分。
    首先按 Asciidoc 标题标记拆分，然后通过合并过小的分块来优化尺寸。
    """

    # 第一步：结构化切分
    # 利用 Asciidoc 的二级标题标记 "\n==" 将文档拆分为多个章节
    chunks = source_text.split("\n==")

    # 初始化变量
    new_chunks = []      # 最终存储合格分块的列表
    chunk_buffer = ""    # 临时缓冲区，用于存那些“太短”的片段
    min_length = 25      # 定义“有效分块”的最小单词数阈值

    # 第二步：贪婪合并逻辑
    for chunk in chunks:
        # 将缓冲区的内容与当前片段合并
        new_buffer = chunk_buffer + chunk  
        
        # 统计当前合并后的单词数
        new_buffer_words = new_buffer.split(" ")  
        
        # 判断长度是否达标
        if len(new_buffer_words) < min_length:  
            # 如果太短（比如只是一个标题或一句话），则存入缓冲区，等待与下一段合并
            chunk_buffer = new_buffer  
        else:
            # 如果长度达标，则认为是一个完整的“知识胶囊”，存入结果列表
            new_chunks.append(new_buffer)  
            # 清空缓冲区，开始收集下一组
            chunk_buffer = ""

    # 第三步：处理残留
    # 如果循环结束后缓冲区里还有内容（最后几个片段没凑够 min_length），也得存起来
    if len(chunk_buffer) > 0:
        new_chunks.append(chunk_buffer)  

    return new_chunks

In [15]:
# 执行混合分块函数，处理之前抓取的 Git 章节文本
mixed_chunks = mixed_chunking(source_text)

# 打印前 3 个分块，用于验证合并逻辑是否生效
for i in range(3):
    # 使用 repr() 打印，这样可以直观看到换行符 \n 和分块的起止位置
    print(f"分块 {i+1}: {repr(mixed_chunks[i])}")

分块 1: "[[what_is_git_section]]= What is Git?\n\nSo, what is Git in a nutshell?\nThis is an important section to absorb, because if you understand what Git is and the fundamentals of how it works, then using Git effectively will probably be much easier for you.\nAs you learn Git, try to clear your mind of the things you may know about other VCSs, such as CVS, Subversion or Perforce -- doing so will help you avoid subtle confusion when using the tool.\nEven though Git's user interface is fairly similar to these other VCSs, Git stores and thinks about information in a very different way, and understanding these differences will help you avoid becoming confused while using it.(((Subversion)))(((Perforce)))\n"
分块 2: "== Snapshots, Not Differences\n\nThe major difference between Git and any other VCS (Subversion and friends included) is the way Git thinks about its data.\nConceptually, most other systems store information as a list of file-based changes.\nThese other systems (CVS, Subversion

这种策略有助于确保分块不会过小，同时仍然利用标题等句法标记来定义边界。在分析了针对单一文本的分块策略后，让我们看看它们在更大规模文本集上的表现如何。

<a id='4'></a>
## 4 - 对真实数据进行分块

---
在本节和接下来的章节中，将提供实际应用中分块的综合示例。你将使用不同的分块方法处理 [Pro Git 书籍](https://git-scm.com/book/en/v2) 的几个章节，然后比较每种方法在搜索任务中的表现。


<a id='4-1'></a>
### 4.1 获取数据

让我们获取整本共 14 章的书籍。

In [16]:
def get_book_text_objects():
    # 初始化一个列表，用于存储最终抓取到的所有文本对象
    text_objs = list()
    
    # GitHub API 基础 URL，指向 Pro Git 书籍的内容目录
    api_base_url = 'https://api.github.com/repos/progit/progit2/contents/book'
    
    # 我们想要抓取的特定章节目录列表
    chapter_urls = ['/01-introduction/sections', '/02-git-basics/sections']

    # 第一层循环：遍历每一个章节目录
    for chapter_url in chapter_urls:
        # 获取该目录下所有文件的元数据列表（JSON 格式）
        # 这一步只是拿到了文件列表，还没有下载具体内容
        response = requests.get(api_base_url + chapter_url)

        # 第二层循环：遍历目录下的每一个具体项（文件或文件夹）
        for file_info in response.json():
            # 过滤逻辑：只处理文件，跳过可能存在的子目录
            if file_info['type'] == 'file':
                # 通过 file_info 中的 'download_url' 获取文件的原始（Raw）文本内容
                # 这就是之前我们手动处理的那个 raw.githubusercontent.com 链接
                file_response = requests.get(file_info['download_url'])

                # 提取元数据：从 URL 中解析出章节标题和文件名
                # 例如：从 ".../01-introduction/..." 中提取出 "01-introduction"
                chapter_title = file_info['download_url'].split('/')[-3]
                filename = file_info['download_url'].split('/')[-1]
                
                # 构建结构化对象
                text_obj = {
                    "body": file_response.text,   # 文件的完整正文
                    "chapter_title": chapter_title, # 所属章节
                    "filename": filename          # 原始文件名
                }
                
                # 将该对象添加到总列表中
                text_objs.append(text_obj)
                
    return text_objs

In [17]:
# 调用之前定义的自动化爬虫函数，从 GitHub 远程抓取《Pro Git》的章节内容
# 执行后，book_text_objs 将会是一个包含 14 个字典对象的列表 (List[Dict])
book_text_objs = get_book_text_objects()

# 每一个元素（对象）都对应书中的一个章节文件，包含以下字段：
# - 'body': 章节的原始文本（Asciidoc 格式）
# - 'chapter_title': 章节所属的目录名（如 01-introduction）
# - 'filename': 具体的文件名（如 what-is-git.asc）

In [18]:
# 打印列表中第一个（索引为 0）字典对象的所有键名（Keys）
# 作用：验证每一个章节对象是否都完整包含了我们定义的字段（body, chapter_title, filename）
print(book_text_objs[0].keys())

# 预期输出结果通常为：
# dict_keys(['body', 'chapter_title', 'filename'])

dict_keys(['body', 'chapter_title', 'filename'])


<a id='4-2'></a>
### 4.2 章节分块

以下分块方法将应用于每个部分：

- **固定长度分块（20% 重叠）：**
  - 每个分块 25 个单词
  - 每个分块 100 个单词

- **可变长度分块**：使用段落标记进行划分

- **混合策略分块**：使用段落标记，并设置最小分块长度为 25 个单词

此外，每个分块都将添加元数据，包括文件名、章节名称和分块编号。

In [19]:
def build_chunk_objs(book_text_obj, chunks):
    """
    根据给定的书籍文本对象及其关联的文本块，构造一个分块对象列表。

    参数:
        book_text_obj (dict): 包含书籍元数据的字典，如 'chapter_title' 和 'filename'。
        chunks (list): 经过切分后的文本块列表。

    返回:
        list: 一个包含多个字典的列表，每个字典代表一个带完整信息的块。
    """
    chunk_objs = list()  # 初始化空列表，用于存储最终的块对象
    
    # 使用 enumerate 遍历 chunks，同时获取索引 i 和内容 c
    for i, c in enumerate(chunks):
        # 为每一个分块创建一个“身份证”字典，整合元数据与内容
        chunk_obj = {
            # 继承来源：记录这段话出自书中的哪一个章节
            "chapter_title": book_text_obj["chapter_title"],  
            
            # 溯源路径：记录原始文件名，方便后续对比查看
            "filename": book_text_obj["filename"],            
            
            # 核心内容：这 100 词左右的具体文本
            "chunk": c,                                       
            
            # 物理位置：记录这是该章节的第几个块，对于恢复上下文或排序非常有用
            "chunk_index": i                                  
        }
        # 将构造好的块对象添加到总列表中
        chunk_objs.append(chunk_obj)

    # 返回这个整齐划一的“知识碎片”列表
    return chunk_objs

In [20]:
# 初始化一个字典，用于存储按不同策略生成的多个分块集合
chunk_obj_sets = dict()

# 第一层循环：遍历之前抓取到的 14 个书籍章节对象
for book_text_obj in book_text_objs:
    text = book_text_obj["body"]  # 提取该章节的原始正文内容

    # 第二层循环：尝试四种不同的分块策略
    # 每一项包含：策略名称 和 调用对应函数后生成的 chunks 列表
    for strategy_name, chunks in [
        # 1. 极细粒度：25个词一组，带 20% 重叠
        ["fixed_size_25", get_chunks_fixed_size_with_overlap(text, 25, 0.2)],
        
        # 2. 标准粒度：100个词一组，带 20% 重叠
        ["fixed_size_100", get_chunks_fixed_size_with_overlap(text, 100, 0.2)],
        
        # 3. 自然段落：按换行符切分，长度不固定
        ["para_chunks", get_chunks_by_paragraph(text)],
        
        # 4. 混合策略：按段落切分，但会自动合并字数少于 25 的碎片
        ["para_chunks_min_25", mixed_chunking(text)]
    ]:
        # 调用之前定义的 build_chunk_objs 函数，将元数据（标题、索引等）注入每个块中
        chunk_objs = build_chunk_objs(book_text_obj, chunks)

        # 整理逻辑：如果该策略名还没在字典里，先初始化一个空列表
        if strategy_name not in chunk_obj_sets.keys():
            chunk_obj_sets[strategy_name] = list()

        # 将当前章节在该策略下生成的所有块，累加到该策略的总列表中
        chunk_obj_sets[strategy_name] += chunk_objs

In [21]:
# 打印字典 chunk_obj_sets 的所有键名（Keys）
print(chunk_obj_sets.keys())

# 预期输出结果：
# dict_keys(['fixed_size_25', 'fixed_size_100', 'para_chunks', 'para_chunks_min_25'])

dict_keys(['fixed_size_25', 'fixed_size_100', 'para_chunks', 'para_chunks_min_25'])


In [22]:
# 选择要查看的分块策略类型：'fixed_size_25', 'fixed_size_100', 'para_chunks' 或 'para_chunks_min_25'
# 你可以手动修改这个字符串，来观察不同切分算法下的结果差异
chunk_type = 'fixed_size_25' 

# 从之前生成的总字典中，提取该策略下的前两个分块对象进行预览
chunk_obj_sets[chunk_type][0:2]

# 作用：
# 1. 策略对比：观察 'fixed_size_25' 是否因为块太小而导致句子被切得支离破碎。
# 2. 结构验证：确认字典中是否正确包含了 chapter_title、filename、chunk 和 chunk_index。
# 3. 边界检查：确认重叠逻辑（Overlap）是否生效（比如分块 1 的结尾和分块 2 的开头是否有重复单词）。

[{'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': '=== About Version Control (((version control))) What is "`version control`", and why should you care? Version control is a system that records changes to a',
  'chunk_index': 0},
 {'chapter_title': '01-introduction',
  'filename': 'about-version-control.asc',
  'chunk': 'that records changes to a file or set of files over time so that you can recall specific versions later. For the examples in this book, you will use software',
  'chunk_index': 1}]

<a id='4-3'></a>
### 4.3 将分块加载到向量数据库中

在本节中，你将重点学习如何将分块加载到向量数据库中。在下方，你将看到有关如何创建向量数据库并向其中加载数据的概要。不过，为了节省时间，在本实验中你将使用一个预先加载好的集合。如果你尚未完成关于 Weaviate API 的练习实验，强烈建议你先去完成，以便更好地理解这一过程！

In [23]:
import os

# 强制清理可能占用 Weaviate 默认端口的残留进程
# 8080/8079 为 REST 端口，50050/50051 为高性能 gRPC 通信端口
# kill_processes_on_ports([8080, 8079, 50050, 50051])

# 设置数据库文件的持久化（存放）路径
# 如果环境变量 COLLECTION_M3 不存在，则默认存放在当前目录 './'
# collection_base = os.getenv('COLLECTION_M3', './')
# persistence_path = os.path.join(collection_base, 'ungraded_lab_2')

# 使用 suppress_subprocess_output 隐藏 Weaviate 启动时的冗余后台日志
with suppress_subprocess_output():
    try:
        # 尝试启动并连接“嵌入式”Weaviate 实例
        # 这种模式不需要你手动运行 Docker，它会随 Python 代码一起启动
        client = weaviate.connect_to_embedded(
            persistence_data_path="./.collections",
            environment_variables={
                "ENABLE_API_BASED_MODULES": "true", # 允许 Weaviate 调用外部推理 API
                "ENABLE_MODULES": 'text2vec-transformers', # 指定使用 Transformer 向量化模块
                # 关键：告诉 Weaviate 向量化服务的地址。
                # 127.0.0.1:5000 指向的就是你之前启动的 Flask BGE 模型服务。
                "TRANSFORMERS_INFERENCE_API":"http://127.0.0.1:5000/", 
            }
        )
    except Exception as e:
        # 如果嵌入式启动失败（例如权限问题），则进入“降级方案”：连接本地已有的 Weaviate 服务
        print(f"无法启动嵌入式 Weaviate: {e}")
        print("尝试连接本地运行的 Weaviate (Port 8079)...")
        try:
            client = weaviate.connect_to_local(port=8079, grpc_port=50050)
        except Exception as e2:
            print(f"连接失败: {e2}")
            print("尝试最后的备用端口 (Port 8080)...")
            client = weaviate.connect_to_local(port=8080, grpc_port=50051)

In [24]:
# 检查 Weaviate 中是否已经存在名为 "chunking_example" 的集合
if not client.collections.exists("chunking_example"):
    # 如果不存在，则开始创建一个新的集合
    collection = client.collections.create(
            name='chunking_example',

            # 配置向量化器（Vectorizer）
            vectorizer_config=[Configure.NamedVectors.text2vec_transformers(
                    name="vector", # 给这个向量起个名字，后续查询时会用到
                    
                    # 告诉 Weaviate：不要把集合名称本身（"chunking_example"）也算进向量里
                    # 如果设为 True，它会把类名拼在文本前面一起向量化
                    vectorize_collection_name = False, 
                    
                    # 关键配置：指定推理服务器的 URL
                    # 这里指向你本地运行 BGE 模型的 Flask 应用（5000 端口）
                    inference_url="http://127.0.0.1:5000", 
                )],

            # 定义数据属性（即数据库的列/字段）
            properties=[  
                # chunk: 存储具体的文本片段
                Property(name="chunk", data_type=DataType.TEXT),
                
                # chapter_title: 存储该片段所属的章节名
                Property(name="chapter_title", data_type=DataType.TEXT),
                
                # filename: 存储原始文件名
                Property(name="filename", data_type=DataType.TEXT),
                
                # chunking_strategy: 存储分块策略名称（如 'fixed_size_100'）
                # tokenization = Tokenization.FIELD 表示将整个字符串视为一个整体（不拆词）
                # 这对于后续根据特定策略进行“精确过滤”非常重要
                Property(name="chunking_strategy", data_type=DataType.TEXT, tokenization=Tokenization.FIELD),
                
                # chunk_index: 存储该片段在原章节中的序号（整数）
                Property(name="chunk_index", data_type=DataType.INT),
            ]
        )
else:
    # 如果集合已经存在，则直接获取该集合的引用，避免重复创建
    collection = client.collections.get("chunking_example")

<div class="alert alert-block alert-warning">  
<b>Note</b> <a class="tocSkip"></a><br> 


```python
# Adding elements in the collection - this insertion should NOT run as the collection is already vectorized for you. 
if len(collection) == 0:
    with collection.batch.fixed_size(batch_size=1, concurrent_requests=20) as batch:
        for chunking_strategy, chunk_objects in tqdm.tqdm(chunk_obj_sets.items()):
            for chunk_obj in chunk_objects:
                chunk_obj["chunking_strategy"] = chunking_strategy
                batch.add_object(
                    properties=chunk_obj,
                    uuid=generate_uuid5(chunk_obj)
                )
```

</div>

In [25]:
if len(collection) == 0:
    with collection.batch.fixed_size(batch_size=1, concurrent_requests=20) as batch:
        for chunking_strategy, chunk_objects in tqdm.tqdm(chunk_obj_sets.items()):
            for chunk_obj in chunk_objects:
                chunk_obj["chunking_strategy"] = chunking_strategy
                batch.add_object(
                    properties=chunk_obj,
                    uuid=generate_uuid5(chunk_obj)
                )

In [26]:
# 打印集合中的对象总数（不带任何过滤条件）
# aggregate.over_all() 是 Weaviate 提供的快速统计工具
print(f"总计对象数: {collection.aggregate.over_all().total_count}")

# 遍历之前定义的四种分块策略名称
for chunking_strategy in chunk_obj_sets.keys():
    # 创建一个过滤器：筛选属性 'chunking_strategy' 等于当前策略名的对象
    # 注意：这里用到了之前定义的 Tokenization.FIELD，确保匹配是精确的
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy) 
    
    # 结合过滤器执行聚合查询，统计该特定策略下的对象数量
    count = collection.aggregate.over_all(filters = where_filter).total_count 
    
    # 输出每种策略对应的分块总数，用于验证导入是否完整
    print(f"策略 {chunking_strategy} 的对象数量: {count}")

总计对象数: 1487
策略 fixed_size_25 的对象数量: 672
策略 fixed_size_100 的对象数量: 173
策略 para_chunks 的对象数量: 549
策略 para_chunks_min_25 的对象数量: 93


<a id='5'></a>
## 5 - 搜索
---
在本节中，你将探索不同分块大小下的语义搜索，从而直观地观察分块大小对信息检索的影响。

In [27]:
# 定义搜索查询语句。
# 尝试搜“Git 的历史”这种宏观问题，或者“远程命令”这种微观问题。
search_string = "history of git"  

# 遍历之前存储在字典中的四种分块策略
for chunking_strategy in chunk_obj_sets.keys():
    # 1. 设置过滤器：确保只在当前指定的策略对应的分块中进行搜索
    # 这能保证对比的公平性（即：看看 25词 vs 100词 谁搜得更准）
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)
    
    # 2. 执行向量搜索（near_text）：
    # - 将 search_string 发送给 Flask API (5000 端口) 转化为向量
    # - 在 Weaviate 中寻找与之最接近的向量
    # - limit = 2 表示每种策略只取前两个最相关的结果
    response = collection.query.near_text(search_string, filters = where_filter, limit = 2)
    
    # 打印该策略下的检索报告
    print(f"针对策略 {chunking_strategy.upper()} 的检索结果：\n")
    
    for i, obj in enumerate(response.objects):
        print(f"===== 分块对象 {i} =====")
        # 打印检索出来的具体文本内容
        print(f"{obj.properties['chunk']}")
        print()

I0000 00:00:1776611606.783048 26968120 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


针对策略 FIXED_SIZE_25 的检索结果：

===== 分块对象 0 =====
=== A Short History of Git As with many great things in life, Git began with a bit of creative destruction and fiery controversy. The

===== 分块对象 1 =====
kernel efficiently (speed and data size) Since its birth in 2005, Git has evolved and matured to be easy to use and yet retain these initial qualities. It's amazingly fast,

针对策略 FIXED_SIZE_100 的检索结果：

===== 分块对象 0 =====
=== A Short History of Git As with many great things in life, Git began with a bit of creative destruction and fiery controversy. The Linux kernel is an open source software project of fairly large scope.(((Linux))) During the early years of the Linux kernel maintenance (1991–2002), changes to the software were passed around as patches and archived files. In 2002, the Linux kernel project began using a proprietary DVCS called BitKeeper.(((BitKeeper))) In 2005, the relationship between the community that developed the Linux kernel and the commercial company that developed B

在这个例子中，查询是一个关注“Git 历史”的宽泛问题。结果显示，较长的分块往往表现更好。经检查发现，虽然 25 个单词的分块在语义相似度上可能与查询紧密匹配，但它们缺乏足够的上下文，无法显著增强读者对该主题的理解。相反，检索到的段落分块——特别是那些最小长度为 25 个单词的分块——提供了全面的信息，有效地帮助读者了解 Git 的历史。

In [28]:
# 定义针对“具体操作”的搜索查询语句
# 这个问题通常对应 git remote add <name> <url> 这一指令
search_string = "how to add the url of a remote repository" 

# 遍历所有分块策略进行横向评测
for chunking_strategy in chunk_obj_sets.keys():
    # 1. 策略过滤器：锁定当前循环中的特定分块方案
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)
    
    # 2. 执行近义搜索（向量搜索）：
    # 系统会寻找那些在向量空间中与“添加远程仓库 URL”意义最接近的片段
    response = collection.query.near_text(search_string, filters = where_filter, limit = 2)
    
    # 打印该策略下的实验报告头信息
    print(f"针对策略 {chunking_strategy.upper()} 的检索结果：\n")
    
    # 遍历返回的两个最相关的分块对象
    for i, obj in enumerate(response.objects):
        print(f"===== 分块对象 {i} =====")
        # 打印检索到的内容，重点观察代码段落是否完整
        print(f"{obj.properties['chunk']}")
        print()

针对策略 FIXED_SIZE_25 的检索结果：

===== 分块对象 0 =====
remote))) To add a new remote Git repository as a shortname you can reference easily, run `git remote add <shortname> <url>`: [source,console] ---- $ git remote origin $ git remote

===== 分块对象 1 =====
manage your remote repositories. Remote repositories are versions of your project that are hosted on the Internet or network somewhere. You can have several of them, each of which generally

针对策略 FIXED_SIZE_100 的检索结果：

===== 分块对象 0 =====
adds the `origin` remote for you. Here's how to add a new remote explicitly.(((git commands, remote))) To add a new remote Git repository as a shortname you can reference easily, run `git remote add <shortname> <url>`: [source,console] ---- $ git remote origin $ git remote add pb https://github.com/paulboone/ticgit $ git remote -v origin https://github.com/schacon/ticgit (fetch) origin https://github.com/schacon/ticgit (push) pb https://github.com/paulboone/ticgit (fetch) pb https://github.com/paulboone/ticgit

在这个例子中，查询更加具体，例如用户想要了解如何添加远程仓库的 URL。与之前的情况不同，25 个单词的分块在这里被证明更有用。由于问题非常具体，Weaviate 能够精准定位到包含最相关段落的分块——即如何添加远程仓库（`git remote add <shortname> <url>`）。

尽管其他结果集中也包含部分此类信息，但重要的是要考虑这些结果将如何被使用和展示。较长的结果可能需要用户付出更多的认知精力来提取相关信息。

<a id='6'></a>
## 6 - 集成到 RAG 系统中
---
现在你已经熟悉了分块，并拥有了一个可以正常运行的集合，让我们看看不同的分块大小如何影响文本生成。让我们使用一个简单的提示词。

In [29]:

# 定义提示词模板字符串
# 这个字符串包含了两个占位符：{search_string} 和 {context}，后续会用真实数据替换它们
PROMPT = "Using this information and only this information, please explain {search_string} in a few short points.\nContext: {context}"

# 核心指令解析：
# 1. "Using this information and only this information": 
#    这是 RAG 的“禁令”。强制要求模型只能使用提供的 context，
#    目的是防止模型产生“幻觉（Hallucination）”或使用过时的训练数据。

# 2. "explain {search_string}": 
#    这里的 {search_string} 是用户最初提出的问题（如：如何添加远程仓库）。

# 3. "\nContext: {context}": 
#    这里的 {context} 将会被你刚才在 Weaviate 中检索到的那 2 个最相关的文本块填充。

In [30]:
# 根据分块策略动态设置检索数量，以补偿不同分块大小带来的信息量差异
n_chunks_by_strat = dict()

# 对于较短的分块（如 25 词），我们需要检索更多的数量（8 个）
# 这样总单词量约为 25 * 8 = 200，保证信息覆盖面足够广
n_chunks_by_strat['fixed_size_25'] = 8
n_chunks_by_strat['para_chunks'] = 8

# 对于较长的分块（如 100 词），我们只需检索较少的数量（2 个）
# 这样总单词量约为 100 * 2 = 200，既保证了信息量，又避免了 Token 浪费
n_chunks_by_strat['fixed_size_100'] = 2
n_chunks_by_strat['para_chunks_min_25'] = 2

# 执行检索增强生成（RAG）流程
search_string = "history of git"  # 设定用户的查询问题

# 循环测试每一种分块策略
for chunking_strategy in chunk_obj_sets.keys():
    # 1. 过滤条件：只从当前测试的策略库中寻找答案
    where_filter = Filter.by_property('chunking_strategy').equal(chunking_strategy)
    
    # 2. 向量搜索：根据不同策略对应的 limit (2 或 8) 检索出最相关的多个分块
    response = collection.query.near_text(
        search_string, 
        filters = where_filter, 
        limit = n_chunks_by_strat[chunking_strategy]
    )
    
    # 3. 构造上下文：将检索到的多个分块文本拼接成一长段文字
    context_string = ""
    for obj in response.objects:
        context_string += obj.properties['chunk'] + '\n'
    
    # 4. 填充提示词：将问题和拼接好的上下文填入 PROMPT 模板
    prompt = PROMPT.format(search_string = search_string, context = context_string)
    
    # 5. 调用大语言模型（LLM）：基于提供的上下文生成最终回答
    response = generate_with_single_input(prompt, role = 'assistant')
    
    # 打印测试结果，方便对比不同策略下的回答质量
    print(f"搜索词: {search_string}")
    print(f"分块策略: {chunking_strategy}:")
    print(f"AI 回答:\n\t{response['content']}")
    print()

搜索词: history of git
分块策略: fixed_size_25:
AI 回答:
	Based on the information provided, here is a short history of Git:

*   **Origins:** Git began with "a bit of creative destruction and fiery controversy."
*   **Birth:** Git was born in 2005.
*   **Evolution:** Since its birth, it has evolved and matured to be easy to use while retaining its initial qualities of being amazingly fast and efficient regarding speed and data size.

搜索词: history of git
分块策略: fixed_size_100:
AI 回答:
	

搜索词: history of git
分块策略: para_chunks:
AI 回答:
	Based on the text provided, here is a short history of Git:

* **Origin:** Git was born in 2005.
* **Beginnings:** Its start involved "a bit of creative destruction and fiery controversy."
* **Evolution:** Since its birth, Git has evolved and matured to be easy to use while retaining its initial qualities.

搜索词: history of git
分块策略: para_chunks_min_25:
AI 回答:
	



In [ ]:
# Don't forget to close the client!
client.close()

: 

恭喜！你已经完成了关于分块 (Chunking) 的练习实验！继续加油！